# Lunar Simulation v000

Inline-only lunar orbital simulation and recovery diagnostic. This notebook uses `ForwardModel` through `LunarCampaign`; it does not write plots or cache artifacts.

## 1. Configuration And Frames

Galactic coordinates are the inertial sky frame. Orbit normals are defined in the J2000 mean ecliptic frame, with shared vernal-equinox ascending node, then transformed to Galactic coordinates. Body rotations map inertial vectors into the crossed-dipole frame.

In [ ]:
import time
import numpy as np
import healpy
import matplotlib.pyplot as plt
from astropy.time import Time
import astropy.units as u
from eigsep_sim import LunarCampaign, LunarRecoveryAdapter
from eigsep_sim.models import T21cmModel
from eigsep_sim.spectral import gsm_eigenmodes, eigenmode_filter

FIGSIZE = (12, 4)

profile = "proposal"  # single switch: use "proposal" for the larger study
profiles = {
    "interactive": {"nside": 16, "nchan": 32, "ntimes": 64, "hours": 4.0},
    "proposal": {"nside": 32, "nchan": 32, "ntimes": 128, "hours": 4.0},
}
p = profiles[profile]
config = {
    "profile": profile,
    "spacecraft": {
        "opening_angle_deg": 90.0, "arm_lengths_m": [6.0, 4.0],
        "arm_masses_kg": [1.0, 2.25], "angular_momentum_direction_gal": [1.0, 0.0, 0.2],
        "spin_period_s": 60.0,
        "attitude_phase_offsets_deg": [0.0, 45.0],
    },
    "orbit": {
        "altitude_km": 100.0, "inclinations_deg": [-15.0, 15.0],
        "ascending_node_lon_deg": 0.0, "equinox": "J2000", "epoch": "2025-01-01",
    },
    "antenna": {"nside": p["nside"], "n_modes": 3},
    "receiver": {"T_rx_K": 100.0},
    "sky": {"nside": p["nside"], "n_modes": min(3, p["nchan"] - 1)},
    "frequency": {"min_mhz": 55.0, "max_mhz": 145.0, "nchan": p["nchan"]},
    "integration": {"duration_hours": p["hours"], "ntimes": p["ntimes"], "attitude_step_s": 10.0},
    "recovery": {"include_receiver_offsets": False, "n_eig_modes": 3},
    "monte_carlo": {"nreal": 8 if profile == "interactive" else 200, "seed": 0},
    "surface": {"T_regolith_K": 300.0, "reflectivity_enabled": False},
    "signal_21cm": {"enabled": True, "model_index": 0},
    "sources": {"earth": {"enabled": False}, "sun": {"enabled": False}},
}
config

## 2. Moon, Orbits, And Galactic Coverage

In [ ]:
campaign = LunarCampaign(config)
result = campaign.run()
fig = plt.figure(figsize=FIGSIZE)
ax = fig.add_subplot(121, projection="3d")
u = np.linspace(0, 2*np.pi, 80)
for normal, color in zip(campaign.orbit_normals_gal, ["C0", "C1"]):
    ref = np.cross(normal, [0, 0, 1])
    if np.linalg.norm(ref) < 1e-6: ref = np.cross(normal, [0, 1, 0])
    ref /= np.linalg.norm(ref); ortho = np.cross(normal, ref)
    xyz = np.cos(u)[:, None]*ref + np.sin(u)[:, None]*ortho
    ax.plot(*xyz.T, color=color)
ax.scatter([0], [0], [0], s=180, color="0.6"); ax.set_title("Moon and circular orbit planes")
plt.subplot(122, projection="mollweide")
visible = result.masks.any(axis=(0, 1)); theta, phi = healpy.pix2ang(campaign.sky.nside, np.where(visible)[0])
plt.scatter(np.pi-phi, np.pi/2-theta, s=8); plt.title("Ever-visible Galactic sky pixels")
plt.tight_layout()

## 3. Torque-Free Tumble Invariants And Arm Coverage

In [ ]:
from eigsep_sim.lunar import angular_momentum_for_spin_period, crossed_rod_inertia, integrate_torque_free
I = crossed_rod_inertia(config["spacecraft"]["arm_lengths_m"], config["spacecraft"]["arm_masses_kg"])
t = np.linspace(0, 60 * config['integration']['duration_hours'], config['integration']['ntimes'])
L = angular_momentum_for_spin_period(I, config["spacecraft"]["angular_momentum_direction_gal"], config["spacecraft"]["spin_period_s"])
tumble = integrate_torque_free(I, L, t)
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(t, tumble["kinetic_energy"] / tumble["kinetic_energy"][0] - 1); ax[0].set_title("Fractional kinetic-energy drift")
axes = tumble["rotations_body_to_gal"].apply(np.broadcast_to(campaign.arm_axes_body[0], (len(t), 3)))
ax[1].scatter(np.arctan2(axes[:,1], axes[:,0]), np.arcsin(axes[:,2]), s=3); ax[1].set_title("Arm-axis pointing coverage")
plt.tight_layout()

## 4. GSM Maps And Injected 21-cm Ensemble

In [ ]:
gsm_plus_signal = campaign.sky.basis.deproject(campaign.sky_coeffs)
freqs_mhz = campaign.freqs_hz / 1e6
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(freqs_mhz, gsm_plus_signal.mean(axis=0), label="GSM + T21")
ax[0].plot(freqs_mhz, campaign.T21cm_K, label="injected T21"); ax[0].legend()
models = T21cmModel()(campaign.freqs_hz)
modes = gsm_eigenmodes(gsm_plus_signal, min(config["recovery"]["n_eig_modes"], len(freqs_mhz)-1))
ax[1].plot(freqs_mhz, eigenmode_filter(models, modes).T, alpha=.15); ax[1].set_title("Eigenmode-filtered signal ensemble")
plt.tight_layout()

## 5. BODY-Frame Dipole Beams

In [ ]:
beam_maps = campaign.beam.basis.deproject(campaign.beam.coeffs)
fig, ax = plt.subplots(1, 2, figsize=FIGSIZE)
for d in range(2):
    healpy.mollview(beam_maps[d, :, len(freqs_mhz)//2], fig=fig.number, sub=(1,2,d+1), title=f"Dipole {d} BODY beam")
plt.tight_layout()

## 6. GAL-Frame Masks, Disk Emission, Beam Sampling, And Weights

In [ ]:
adapter = LunarRecoveryAdapter(campaign, result)
weights = adapter.beam_weights(0, len(freqs_mhz)//2)
fig, ax = plt.subplots(1, 2, figsize=FIGSIZE)
healpy.mollview(result.masks[0,0], fig=fig.number, sub=(1,2,1), title="GAL sky mask: spacecraft 0")
healpy.mollview(weights[0,0], fig=fig.number, sub=(1,2,2), title="Normalized integration weights")
plt.tight_layout()

## 7. Full Versus Reduced JAX Geometry

In [ ]:
fwd = campaign.forward_models[0]; times = result.times; body_rots = result.body_rots[0]
t0=time.perf_counter(); full=fwd.precompute_geometry(times=times, body_rots=body_rots); full_dt=time.perf_counter()-t0
sky_mask=fwd.build_sky_mask(times=times); t0=time.perf_counter(); reduced=fwd.precompute_geometry(times=times, body_rots=body_rots, sky_mask=sky_mask); reduced_dt=time.perf_counter()-t0
T_full=np.asarray(fwd.simulate(campaign.sky_coeffs, campaign.beam.coeffs, geom=full)); T_reduced=np.asarray(fwd.simulate(campaign.sky_coeffs, campaign.beam.coeffs, geom=reduced))
print({"max_abs_K": float(np.max(np.abs(T_full-T_reduced))), "full_s": full_dt, "reduced_s": reduced_dt, "pixels_kept": int(sky_mask.sum())})

## 8. Noiseless And Noisy Monopole Recovery

In [ ]:
fi=len(freqs_mhz)//2; sky_map = campaign.sky.basis.deproject(campaign.sky_coeffs)[:, fi]
adapter.verify_noiseless(sky_map, config["surface"]["T_regolith_K"], fi, rtol=5e-5, atol=5e-4)
A=adapter.build_design_matrix(fi); y=adapter.truth_vector(fi)
rng=np.random.default_rng(config["monte_carlo"]["seed"]); sigma=0.05
fits=[np.linalg.lstsq(A, y+rng.normal(0,sigma,len(y)), rcond=1e-6)[0] for _ in range(config["monte_carlo"]["nreal"])]
print({"design_shape": A.shape, "disk_truth_K": 300.0, "disk_fit_mean_K": float(np.mean(fits,axis=0)[-1]), "channel_noise_K": sigma})
all_models = T21cmModel()(campaign.freqs_hz)
filtered_models = eigenmode_filter(all_models, modes)
filtered_injected = eigenmode_filter(campaign.T21cm_K, modes)
chi2 = np.sum(((filtered_models - filtered_injected) / sigma) ** 2, axis=1)
best_models = np.argsort(chi2)[:5]
combined_snr = float(np.linalg.norm(filtered_injected / sigma))
print({"best_model_indices": best_models.tolist(), "combined_snr": combined_snr})

## 9. Recovered GSM Map And Angular-Power Sensitivity

In [ ]:
x=np.linalg.lstsq(A, y, rcond=1e-6)[0]; recovered=x[:-1]; residual=(recovered-sky_map)/sky_map
fig, ax=plt.subplots(1,2,figsize=(10,3))
healpy.mollview(recovered, fig=fig.number, sub=(1,2,1), cmap='plasma', title="Recovered GSM map")
healpy.mollview(residual, fig=fig.number, sub=(1,2,2), cmap='bwr', title="Fractional residual")
plt.tight_layout(); print("Residual angular power:", healpy.anafast(np.nan_to_num(residual))[:5])

## 10. Pointing Knowledge And Noise Budget

In [ ]:
knowledge_deg=np.logspace(-3,0,20); thermal_mK=50.0; pointing_mK=122.29*knowledge_deg
plt.figure(figsize=(6,3)); plt.loglog(knowledge_deg, np.hypot(thermal_mK, pointing_mK), label="combined"); plt.loglog(knowledge_deg, pointing_mK, label="pointing"); plt.axhline(thermal_mK, color="k", ls="--", label="thermal"); plt.legend(); plt.xlabel("beam-orientation knowledge [deg]"); plt.ylabel("noise / leakage [mK]"); plt.tight_layout()

## 11. Deferred Physics Extension Points

`LunarOrbit` owns lunar occultation geometry through `above_horizon()` and `above_horizon_stack()`, while uniform regolith brightness is configured with `occultation_temperature_K`. A future Moon-fixed HEALPix temperature model should extend the observer-owned occultation geometry instead of adding a parallel surface-model API. `LunarRecoveryAdapter.source_registry` reserves disabled Earth and Sun columns. In v000, spatial regolith temperatures, reflectivity, Earth emission, and Sun emission remain disabled explicitly.